In [7]:
# 2차 데이터 개선 1번 셀: previous-day feature dataset 가능성 확인
# 목적:
# - sleep target calendar_date 기준으로 같은 날짜 feature를 쓰지 않고, 전날 feature만 사용할 수 있는지 확인한다.
# - participant별로 feature columns를 1일 shift한다.
# - target 날짜와 feature 날짜가 정확히 하루 차이인지 검증한다.
# - 아직 파일 저장은 하지 않는다.

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_PATH = PROCESSED_DIR / "modeling_dataset_encoded.csv"
FEATURE_PATH = PROCESSED_DIR / "encoded_feature_columns.csv"
SPLIT_PATH = PROCESSED_DIR / "deep_learning_sequences" / "deep_learning_participant_split_assignments.csv"

TARGET_COL = "good_sleep_label"
ID_COL = "participant_object_id"
DATE_COL = "calendar_date"
SPLIT_COL = "deep_learning_split"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
feature_df = pd.read_csv(FEATURE_PATH, encoding="utf-8-sig")
split_df = pd.read_csv(SPLIT_PATH, encoding="utf-8-sig")

feature_cols = feature_df["feature"].tolist()

split_df = split_df[[ID_COL, SPLIT_COL]].drop_duplicates()
df = df.merge(split_df, on=ID_COL, how="left")

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)

print("original df shape:", df.shape)
print("feature count:", len(feature_cols))
print(df[SPLIT_COL].value_counts(dropna=False))


def create_previous_day_frame(data, feature_columns):
    base_cols = [ID_COL, DATE_COL, TARGET_COL, SPLIT_COL]

    target_frame = data[base_cols].copy()
    target_frame = target_frame.rename(columns={DATE_COL: "target_date"})

    feature_frame = data[[ID_COL, DATE_COL] + feature_columns].copy()
    feature_frame = feature_frame.rename(columns={DATE_COL: "feature_date"})

    feature_frame["target_date"] = feature_frame["feature_date"] + pd.Timedelta(days=1)

    previous_day = target_frame.merge(
        feature_frame,
        on=[ID_COL, "target_date"],
        how="left",
    )

    previous_day["feature_lag_days"] = (
        previous_day["target_date"] - previous_day["feature_date"]
    ).dt.days

    return previous_day


previous_day_df = create_previous_day_frame(df, feature_cols)

coverage_summary = {
    "original_rows": len(df),
    "previous_day_rows": len(previous_day_df),
    "rows_with_previous_day_features": int(previous_day_df["feature_date"].notna().sum()),
    "rows_without_previous_day_features": int(previous_day_df["feature_date"].isna().sum()),
    "previous_day_coverage_rate": float(previous_day_df["feature_date"].notna().mean()),
}

print('coverage_summary')
print(coverage_summary)

lag_counts = previous_day_df["feature_lag_days"].value_counts(dropna=False).sort_index()
print("lag_counts")
display(lag_counts)

split_coverage = (
    previous_day_df
    .groupby(SPLIT_COL)
    .agg(
        rows=("target_date", "size"),
        rows_with_previous_day_features=("feature_date", lambda s: int(s.notna().sum())),
        coverage_rate=("feature_date", lambda s: float(s.notna().mean())),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
    )
    .reset_index()
)

display(split_coverage)

usable_previous_day_df = previous_day_df[previous_day_df["feature_date"].notna()].copy()

print("usable_previous_day_df shape:", usable_previous_day_df.shape)
print("usable split counts:")
print(usable_previous_day_df[SPLIT_COL].value_counts())

participant_coverage = (
    previous_day_df
    .assign(has_previous_day=previous_day_df["feature_date"].notna())
    .groupby([SPLIT_COL, ID_COL], as_index=False)
    .agg(
        rows=("target_date", "size"),
        usable_rows=("has_previous_day", "sum"),
        coverage_rate=("has_previous_day", "mean"),
        target_mean=(TARGET_COL, "mean"),
    )
)

display(
    participant_coverage
    .groupby(SPLIT_COL)
    .agg(
        participants=(ID_COL, "nunique"),
        mean_rows=("rows", "mean"),
        mean_usable_rows=("usable_rows", "mean"),
        min_usable_rows=("usable_rows", "min"),
        max_usable_rows=("usable_rows", "max"),
        mean_coverage_rate=("coverage_rate", "mean"),
    )
    .reset_index()
)

original df shape: (3551, 201)
feature count: 197
deep_learning_split
train         2425
test           607
validation     519
Name: count, dtype: int64
coverage_summary
{'original_rows': 3551, 'previous_day_rows': 3551, 'rows_with_previous_day_features': 3116, 'rows_without_previous_day_features': 435, 'previous_day_coverage_rate': 0.8774992959729654}
lag_counts


feature_lag_days
1.0    3116
NaN     435
Name: count, dtype: int64

,deep_learning_split,rows,rows_with_previous_day_features,coverage_rate,participants,target_mean
0,test,607,524,0.863262,14,0.369028
1,train,2425,2135,0.880412,44,0.400000
2,validation,519,457,0.880539,11,0.393064


usable_previous_day_df shape: (3116, 203)
usable split counts:
deep_learning_split
train         2135
test           524
validation     457
Name: count, dtype: int64


,deep_learning_split,participants,mean_rows,mean_usable_rows,min_usable_rows,max_usable_rows,mean_coverage_rate
0,test,14,43.357143,37.428571,0,79,0.697417
1,train,44,55.113636,48.522727,0,200,0.780599
2,validation,11,47.181818,41.545455,0,81,0.752945


In [8]:
# 2차 데이터 개선 2번 셀: previous-day modeling dataset 저장
# 목적:
# - target_date의 label은 그대로 두고, feature는 target_date - 1일 값만 사용한다.
# - same-date feature 사용 위험을 줄인 previous-day dataset을 만든다.
# - feature column 이름은 기존과 동일하게 유지한다.
# - feature_date, feature_lag_days를 함께 저장해서 추적 가능하게 한다.

PREVIOUS_DAY_DIR = PROCESSED_DIR / "deep_learning_previous_day"
PREVIOUS_DAY_DIR.mkdir(parents=True, exist_ok=True)

previous_day_usable = previous_day_df[previous_day_df["feature_date"].notna()].copy()

previous_day_usable = previous_day_usable.rename(columns={"target_date": DATE_COL})

ordered_cols = [
    ID_COL,
    DATE_COL,
    "feature_date",
    "feature_lag_days",
    TARGET_COL,
    SPLIT_COL,
] + feature_cols

previous_day_usable = previous_day_usable[ordered_cols].sort_values(
    [ID_COL, DATE_COL]
).reset_index(drop=True)

previous_day_dataset_path = PREVIOUS_DAY_DIR / "modeling_dataset_previous_day_encoded.csv"
previous_day_feature_path = PREVIOUS_DAY_DIR / "previous_day_feature_columns.csv"

previous_day_usable.to_csv(
    previous_day_dataset_path,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame({"feature": feature_cols}).to_csv(
    previous_day_feature_path,
    index=False,
    encoding="utf-8-sig",
)

previous_day_summary = {
    "path": str(previous_day_dataset_path),
    "rows": len(previous_day_usable),
    "features": len(feature_cols),
    "participants": previous_day_usable[ID_COL].nunique(),
    "min_target_date": str(previous_day_usable[DATE_COL].min().date()),
    "max_target_date": str(previous_day_usable[DATE_COL].max().date()),
    "feature_lag_days_unique": sorted(previous_day_usable["feature_lag_days"].dropna().unique().tolist()),
}

print(previous_day_summary)

display(previous_day_usable[SPLIT_COL].value_counts().rename("rows"))

display(
    previous_day_usable
    .groupby(SPLIT_COL)
    .agg(
        rows=(TARGET_COL, "size"),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        min_target_date=(DATE_COL, "min"),
        max_target_date=(DATE_COL, "max"),
    )
    .reset_index()
)

{'path': 'c:\\workSpace\\DeepLearnin_sleep\\data\\processed\\deep_learning_previous_day\\modeling_dataset_previous_day_encoded.csv', 'rows': 3116, 'features': 197, 'participants': 63, 'min_target_date': '2021-05-25', 'max_target_date': '2022-01-22', 'feature_lag_days_unique': [1.0]}


deep_learning_split
train         2135
test           524
validation     457
Name: rows, dtype: int64

,deep_learning_split,rows,participants,target_mean,min_target_date,max_target_date
0,test,524,12,0.379771,2021-05-25,2022-01-22
1,train,2135,41,0.405621,2021-05-25,2022-01-21
2,validation,457,10,0.393873,2021-05-25,2022-01-21


In [9]:
# 2차 데이터 개선 3번 셀: previous-day feature subset별 sequence tensor 생성
# 목적:
# - previous-day encoded dataset을 사용해서 subset별 sequence tensor를 만든다.
# - 기존 Phase 1A와 같은 subset 정의를 재사용한다.
# - 출력은 deep_learning_previous_day/sequences_by_subset 아래에 저장한다.

from sklearn.preprocessing import StandardScaler
import joblib
import json

PREVIOUS_DAY_DIR = PROCESSED_DIR / "deep_learning_previous_day"
PREVIOUS_DAY_DATA_PATH = PREVIOUS_DAY_DIR / "modeling_dataset_previous_day_encoded.csv"

SUBSET_DIR = PROCESSED_DIR / "deep_learning_feature_subsets"
PREVIOUS_DAY_SEQUENCE_ROOT = PREVIOUS_DAY_DIR / "sequences_by_subset"

WINDOW_SIZES = [7, 14, 30]

previous_df = pd.read_csv(PREVIOUS_DAY_DATA_PATH, encoding="utf-8-sig")
previous_df[DATE_COL] = pd.to_datetime(previous_df[DATE_COL])
previous_df["feature_date"] = pd.to_datetime(previous_df["feature_date"])
previous_df = previous_df.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)

subset_paths = sorted(SUBSET_DIR.glob("*_features.csv"))

feature_subsets = {}

for path in subset_paths:
    subset_name = path.name.replace("_features.csv", "")
    subset_df = pd.read_csv(path, encoding="utf-8-sig")
    feature_subsets[subset_name] = subset_df["feature"].tolist()

print("previous_df shape:", previous_df.shape)
print("subsets:", {k: len(v) for k, v in feature_subsets.items()})


def scale_previous_subset(data, features):
    scaled = data.copy()
    train_mask = scaled[SPLIT_COL] == "train"

    scaler = StandardScaler()
    scaler.fit(scaled.loc[train_mask, features])

    scaled.loc[:, features] = scaler.transform(scaled[features]).astype(np.float32)

    return scaled, scaler


def build_previous_windows_for_split(data, features, split_name, window_size):
    split_df = data[data[SPLIT_COL] == split_name].copy()
    split_df = split_df.sort_values([ID_COL, DATE_COL])

    X_sequence = []
    X_mlp_flattened = []
    X_mlp_current_day = []
    y = []
    participant_ids = []
    window_start_dates = []
    window_end_dates = []
    feature_start_dates = []
    feature_end_dates = []

    for participant_id, group in split_df.groupby(ID_COL, sort=False):
        group = group.sort_values(DATE_COL).reset_index(drop=True)

        feature_values = group[features].to_numpy(dtype=np.float32)
        target_values = group[TARGET_COL].to_numpy(dtype=np.int64)
        target_dates = group[DATE_COL].to_numpy()
        feature_dates = group["feature_date"].to_numpy()

        for end_idx in range(window_size - 1, len(group)):
            start_idx = end_idx - window_size + 1

            target_window_dates = pd.to_datetime(target_dates[start_idx : end_idx + 1])
            feature_window_dates = pd.to_datetime(feature_dates[start_idx : end_idx + 1])

            expected_target_dates = pd.date_range(
                start=target_window_dates[0],
                periods=window_size,
                freq="D",
            )

            expected_feature_dates = expected_target_dates - pd.Timedelta(days=1)

            target_contiguous = np.array_equal(
                target_window_dates.to_numpy(),
                expected_target_dates.to_numpy(),
            )

            feature_contiguous = np.array_equal(
                feature_window_dates.to_numpy(),
                expected_feature_dates.to_numpy(),
            )

            if not target_contiguous or not feature_contiguous:
                continue

            window_x = feature_values[start_idx : end_idx + 1]

            X_sequence.append(window_x)
            X_mlp_flattened.append(window_x.reshape(-1))
            X_mlp_current_day.append(window_x[-1])
            y.append(target_values[end_idx])
            participant_ids.append(participant_id)
            window_start_dates.append(str(pd.Timestamp(target_window_dates[0]).date()))
            window_end_dates.append(str(pd.Timestamp(target_window_dates[-1]).date()))
            feature_start_dates.append(str(pd.Timestamp(feature_window_dates[0]).date()))
            feature_end_dates.append(str(pd.Timestamp(feature_window_dates[-1]).date()))

    if X_sequence:
        X_sequence = np.stack(X_sequence).astype(np.float32)
        X_mlp_flattened = np.stack(X_mlp_flattened).astype(np.float32)
        X_mlp_current_day = np.stack(X_mlp_current_day).astype(np.float32)
        y = np.asarray(y, dtype=np.int64)
    else:
        X_sequence = np.empty((0, window_size, len(features)), dtype=np.float32)
        X_mlp_flattened = np.empty((0, window_size * len(features)), dtype=np.float32)
        X_mlp_current_day = np.empty((0, len(features)), dtype=np.float32)
        y = np.empty((0,), dtype=np.int64)

    return {
        "X_sequence": X_sequence,
        "X_mlp_flattened": X_mlp_flattened,
        "X_mlp_current_day": X_mlp_current_day,
        "y": y,
        "participant_object_id": np.asarray(participant_ids, dtype=object),
        "window_start_date": np.asarray(window_start_dates, dtype=object),
        "window_end_date": np.asarray(window_end_dates, dtype=object),
        "feature_start_date": np.asarray(feature_start_dates, dtype=object),
        "feature_end_date": np.asarray(feature_end_dates, dtype=object),
        "feature_names": np.asarray(features, dtype=object),
    }


summary_rows = []

PREVIOUS_DAY_SEQUENCE_ROOT.mkdir(parents=True, exist_ok=True)

for subset_name, features in feature_subsets.items():
    print(f"\n=== subset: {subset_name} ({len(features)} features) ===")

    scaled_df, scaler = scale_previous_subset(previous_df, features)

    subset_root = PREVIOUS_DAY_SEQUENCE_ROOT / subset_name
    subset_root.mkdir(parents=True, exist_ok=True)

    joblib.dump(scaler, subset_root / "standard_scaler.joblib")

    pd.DataFrame({"feature": features}).to_csv(
        subset_root / "feature_columns.csv",
        index=False,
        encoding="utf-8-sig",
    )

    for window_size in WINDOW_SIZES:
        window_root = subset_root / f"window_{window_size}"
        window_root.mkdir(parents=True, exist_ok=True)

        all_sample_index = []

        for split_name in ["train", "validation", "test"]:
            arrays = build_previous_windows_for_split(
                scaled_df,
                features,
                split_name,
                window_size,
            )

            npz_path = window_root / f"{split_name}.npz"
            np.savez_compressed(npz_path, **arrays)

            sample_index = pd.DataFrame({
                "window": window_size,
                "split": split_name,
                "participant_object_id": arrays["participant_object_id"],
                "window_start_date": arrays["window_start_date"],
                "window_end_date": arrays["window_end_date"],
                "feature_start_date": arrays["feature_start_date"],
                "feature_end_date": arrays["feature_end_date"],
                "good_sleep_label": arrays["y"],
            })

            all_sample_index.append(sample_index)

            y = arrays["y"]

            summary_rows.append({
                "subset_name": subset_name,
                "window": window_size,
                "split": split_name,
                "samples": int(len(y)),
                "participants": int(len(set(arrays["participant_object_id"]))),
                "features": int(len(features)),
                "sequence_shape": " x ".join(map(str, arrays["X_sequence"].shape)),
                "mlp_flattened_shape": " x ".join(map(str, arrays["X_mlp_flattened"].shape)),
                "mlp_current_day_shape": " x ".join(map(str, arrays["X_mlp_current_day"].shape)),
                "class_0_samples": int((y == 0).sum()),
                "class_1_samples": int((y == 1).sum()),
                "good_sleep_label_mean": float(y.mean()) if len(y) else np.nan,
                "tensor_path": str(npz_path),
            })

            print(
                f"window={window_size:>2} split={split_name:<10} "
                f"samples={len(y):>4} shape={arrays['X_sequence'].shape}"
            )

        pd.concat(all_sample_index, ignore_index=True).to_csv(
            window_root / "sample_index.csv",
            index=False,
            encoding="utf-8-sig",
        )

    metadata = {
        "subset_name": subset_name,
        "feature_count": len(features),
        "windows": WINDOW_SIZES,
        "target_col": TARGET_COL,
        "id_col": ID_COL,
        "date_col": DATE_COL,
        "split_col": SPLIT_COL,
        "source_data": str(PREVIOUS_DAY_DATA_PATH),
        "feature_policy": "target_date uses features from target_date_minus_1",
    }

    with open(subset_root / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

previous_sequence_summary = pd.DataFrame(summary_rows)

previous_sequence_summary_path = PREVIOUS_DAY_SEQUENCE_ROOT / "sequence_tensor_summary_previous_day_by_subset.csv"
previous_sequence_summary.to_csv(
    previous_sequence_summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("\nsaved:", previous_sequence_summary_path)
display(previous_sequence_summary)

previous_df shape: (3116, 203)
subsets: {'daytime_context_survey': 115, 'daytime_only': 19, 'full_current': 197, 'no_high_sleep_session': 127, 'no_high_sleep_session_no_stress': 115}

=== subset: daytime_context_survey (115 features) ===
window= 7 split=train      samples=1303 shape=(1303, 7, 115)
window= 7 split=validation samples= 245 shape=(245, 7, 115)
window= 7 split=test       samples= 296 shape=(296, 7, 115)
window=14 split=train      samples= 887 shape=(887, 14, 115)
window=14 split=validation samples= 136 shape=(136, 14, 115)
window=14 split=test       samples= 206 shape=(206, 14, 115)
window=30 split=train      samples= 415 shape=(415, 30, 115)
window=30 split=validation samples=  52 shape=(52, 30, 115)
window=30 split=test       samples= 101 shape=(101, 30, 115)

=== subset: daytime_only (19 features) ===
window= 7 split=train      samples=1303 shape=(1303, 7, 19)
window= 7 split=validation samples= 245 shape=(245, 7, 19)
window= 7 split=test       samples= 296 shape=(296, 7

,subset_name,window,split,samples,participants,features,sequence_shape,mlp_flattened_shape,mlp_current_day_shape,class_0_samples,class_1_samples,good_sleep_label_mean,tensor_path
0,daytime_context_survey,7,train,1303,33,115,1303 x 7 x 115,1303 x 805,1303 x 115,779,524,0.402149,c:\workSpace\DeepLearnin_sleep\data\processed\...
1,daytime_context_survey,7,validation,245,9,115,245 x 7 x 115,245 x 805,245 x 115,153,92,0.375510,c:\workSpace\DeepLearnin_sleep\data\processed\...
2,daytime_context_survey,7,test,296,9,115,296 x 7 x 115,296 x 805,296 x 115,173,123,0.415541,c:\workSpace\DeepLearnin_sleep\data\processed\...
3,daytime_context_survey,14,train,887,27,115,887 x 14 x 115,887 x 1610,887 x 115,553,334,0.376550,c:\workSpace\DeepLearnin_sleep\data\processed\...
4,daytime_context_survey,14,validation,136,7,115,136 x 14 x 115,136 x 1610,136 x 115,88,48,0.352941,c:\workSpace\DeepLearnin_sleep\data\processed\...
5,daytime_context_survey,14,test,206,5,115,206 x 14 x 115,206 x 1610,206 x 115,109,97,0.470874,c:\workSpace\DeepLearnin_sleep\data\processed\...
6,daytime_context_survey,30,train,415,17,115,415 x 30 x 115,415 x 3450,415 x 115,271,144,0.346988,c:\workSpace\DeepLearnin_sleep\data\processed\...
7,daytime_context_survey,30,validation,52,3,115,52 x 30 x 115,52 x 3450,52 x 115,40,12,0.230769,c:\workSpace\DeepLearnin_sleep\data\processed\...
8,daytime_context_survey,30,test,101,4,115,101 x 30 x 115,101 x 3450,101 x 115,41,60,0.594059,c:\workSpace\DeepLearnin_sleep\data\processed\...
9,daytime_only,7,train,1303,33,19,1303 x 7 x 19,1303 x 133,1303 x 19,779,524,0.402149,c:\workSpace\DeepLearnin_sleep\data\processed\...


In [ ]:
# 2차 데이터 개선 4번 셀: previous-day tensor 검증 + Phase 2A grid 생성
# 목적:
# - previous-day sequence tensor 파일들이 정상인지 검증한다.
# - 같은-date Phase 1A에서 유망했던 조합 중심으로 previous-day Phase 2A grid를 만든다.
# - 우선 daytime_only 중심, window 7/14 중심으로 간다.

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PREVIOUS_DAY_DIR = PROCESSED_DIR / "deep_learning_previous_day"
PREVIOUS_DAY_SEQUENCE_ROOT = PREVIOUS_DAY_DIR / "sequences_by_subset"
PREVIOUS_DAY_SUMMARY_PATH = (
    PREVIOUS_DAY_SEQUENCE_ROOT / "sequence_tensor_summary_previous_day_by_subset.csv"
)

EXPERIMENT_DIR = PREVIOUS_DAY_DIR / "experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

previous_sequence_summary = pd.read_csv(PREVIOUS_DAY_SUMMARY_PATH, encoding="utf-8-sig")

print("summary shape:", previous_sequence_summary.shape)
display(previous_sequence_summary)

validation_rows = []

for _, row in previous_sequence_summary.iterrows():
    subset_name = row["subset_name"]
    window = int(row["window"])
    split = row["split"]

    subset_root = PREVIOUS_DAY_SEQUENCE_ROOT / subset_name
    window_root = subset_root / f"window_{window}"

    npz_path = window_root / f"{split}.npz"
    feature_path = subset_root / "feature_columns.csv"
    sample_index_path = window_root / "sample_index.csv"

    arrays = np.load(npz_path, allow_pickle=True)
    feature_df = pd.read_csv(feature_path, encoding="utf-8-sig")
    sample_index = pd.read_csv(sample_index_path, encoding="utf-8-sig")

    X_sequence = arrays["X_sequence"]
    X_mlp_current_day = arrays["X_mlp_current_day"]
    X_mlp_flattened = arrays["X_mlp_flattened"]
    y = arrays["y"]
    feature_names = arrays["feature_names"]

    sample_index_split = sample_index[sample_index["split"] == split]

    validation_rows.append({
        "subset_name": subset_name,
        "window": window,
        "split": split,
        "npz_exists": npz_path.exists(),
        "feature_csv_count": len(feature_df),
        "feature_npz_count": len(feature_names),
        "summary_features": int(row["features"]),
        "samples_npz": len(y),
        "samples_summary": int(row["samples"]),
        "samples_index": len(sample_index_split),
        "X_sequence_shape": str(X_sequence.shape),
        "has_nan": bool(np.isnan(X_sequence).any()),
        "has_inf": bool(np.isinf(X_sequence).any()),
        "feature_count_match": len(feature_df) == len(feature_names) == int(row["features"]),
        "sample_count_match": len(y) == int(row["samples"]) == len(sample_index_split),
        "sequence_shape_match": X_sequence.shape == (len(y), window, len(feature_names)),
        "mlp_current_day_shape_match": X_mlp_current_day.shape == (len(y), len(feature_names)),
        "mlp_flattened_shape_match": X_mlp_flattened.shape == (len(y), window * len(feature_names)),
    })

validation_df = pd.DataFrame(validation_rows)

problem_cols = [
    "npz_exists",
    "feature_count_match",
    "sample_count_match",
    "sequence_shape_match",
    "mlp_current_day_shape_match",
    "mlp_flattened_shape_match",
]

problems = validation_df[
    (validation_df[problem_cols] == False).any(axis=1)
    | validation_df["has_nan"]
    | validation_df["has_inf"]
]

print("problem rows:", len(problems))
display(problems)

compact_summary = (
    validation_df
    .groupby(["subset_name", "window"], as_index=False)
    .agg(
        feature_count=("feature_npz_count", "first"),
        train_samples=("samples_npz", lambda s: int(s.iloc[0])),
        all_feature_count_match=("feature_count_match", "all"),
        all_sample_count_match=("sample_count_match", "all"),
        all_sequence_shape_match=("sequence_shape_match", "all"),
        any_nan=("has_nan", "any"),
        any_inf=("has_inf", "any"),
    )
)

display(compact_summary)


# Phase 2A는 previous-day 데이터에서 가장 중요한 조합부터 실행한다.
# Phase 1A 결과상 daytime_only가 가장 안정적이었고, window 7/14가 participant 수 측면에서 더 안전했다.
phase_2a_rows = []

PHASE_2A_SUBSETS = [
    "daytime_only",
    "no_high_sleep_session",
]

PHASE_2A_WINDOWS = [
    7,
    14,
]

PHASE_2A_MODELS = [
    "mlp_current_day",
    "gru",
    "bilstm",
    "cnn_1d",
]

for subset_name in PHASE_2A_SUBSETS:
    for window in PHASE_2A_WINDOWS:
        for model_family in PHASE_2A_MODELS:
            phase_2a_rows.append({
                "phase": "phase_2a_previous_day_priority",
                "subset_name": subset_name,
                "window": window,
                "model_family": model_family,
                "run": True,
                "selection_metric": "validation_balanced_accuracy",
                "test_policy": "validation ranking 이후 확인",
            })

phase_2a_grid = pd.DataFrame(phase_2a_rows)
phase_2a_grid["experiment_id"] = [
    f"phase2a_{i:03d}" for i in range(len(phase_2a_grid))
]

phase_2a_grid = phase_2a_grid[
    [
        "experiment_id",
        "phase",
        "subset_name",
        "window",
        "model_family",
        "run",
        "selection_metric",
        "test_policy",
    ]
]

phase_2a_grid_path = EXPERIMENT_DIR / "phase_2a_previous_day_experiment_grid.csv"
phase_2a_grid.to_csv(phase_2a_grid_path, index=False, encoding="utf-8-sig")

print("saved:", phase_2a_grid_path)
print("phase_2a experiments:", len(phase_2a_grid))

display(phase_2a_grid)

display(
    phase_2a_grid
    .groupby(["subset_name", "window"], as_index=False)
    .agg(
        experiments=("experiment_id", "count"),
        models=("model_family", lambda s: sorted(s.unique())),
    )
)

summary shape: (45, 13)


,subset_name,window,split,samples,participants,features,sequence_shape,mlp_flattened_shape,mlp_current_day_shape,class_0_samples,class_1_samples,good_sleep_label_mean,tensor_path
0,daytime_context_survey,7,train,1303,33,115,1303 x 7 x 115,1303 x 805,1303 x 115,779,524,0.402149,c:\workSpace\DeepLearnin_sleep\data\processed\...
1,daytime_context_survey,7,validation,245,9,115,245 x 7 x 115,245 x 805,245 x 115,153,92,0.375510,c:\workSpace\DeepLearnin_sleep\data\processed\...
2,daytime_context_survey,7,test,296,9,115,296 x 7 x 115,296 x 805,296 x 115,173,123,0.415541,c:\workSpace\DeepLearnin_sleep\data\processed\...
3,daytime_context_survey,14,train,887,27,115,887 x 14 x 115,887 x 1610,887 x 115,553,334,0.376550,c:\workSpace\DeepLearnin_sleep\data\processed\...
4,daytime_context_survey,14,validation,136,7,115,136 x 14 x 115,136 x 1610,136 x 115,88,48,0.352941,c:\workSpace\DeepLearnin_sleep\data\processed\...
5,daytime_context_survey,14,test,206,5,115,206 x 14 x 115,206 x 1610,206 x 115,109,97,0.470874,c:\workSpace\DeepLearnin_sleep\data\processed\...
6,daytime_context_survey,30,train,415,17,115,415 x 30 x 115,415 x 3450,415 x 115,271,144,0.346988,c:\workSpace\DeepLearnin_sleep\data\processed\...
7,daytime_context_survey,30,validation,52,3,115,52 x 30 x 115,52 x 3450,52 x 115,40,12,0.230769,c:\workSpace\DeepLearnin_sleep\data\processed\...
8,daytime_context_survey,30,test,101,4,115,101 x 30 x 115,101 x 3450,101 x 115,41,60,0.594059,c:\workSpace\DeepLearnin_sleep\data\processed\...
9,daytime_only,7,train,1303,33,19,1303 x 7 x 19,1303 x 133,1303 x 19,779,524,0.402149,c:\workSpace\DeepLearnin_sleep\data\processed\...


problem rows: 0


,subset_name,window,split,npz_exists,feature_csv_count,feature_npz_count,summary_features,samples_npz,samples_summary,samples_index,X_sequence_shape,has_nan,has_inf,feature_count_match,sample_count_match,sequence_shape_match,mlp_current_day_shape_match,mlp_flattened_shape_match


,subset_name,window,feature_count,train_samples,all_feature_count_match,all_sample_count_match,all_sequence_shape_match,any_nan,any_inf
0,daytime_context_survey,7,115,1303,True,True,True,False,False
1,daytime_context_survey,14,115,887,True,True,True,False,False
2,daytime_context_survey,30,115,415,True,True,True,False,False
3,daytime_only,7,19,1303,True,True,True,False,False
4,daytime_only,14,19,887,True,True,True,False,False
5,daytime_only,30,19,415,True,True,True,False,False
6,full_current,7,197,1303,True,True,True,False,False
7,full_current,14,197,887,True,True,True,False,False
8,full_current,30,197,415,True,True,True,False,False
9,no_high_sleep_session,7,127,1303,True,True,True,False,False


saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_previous_day_experiment_grid.csv
phase_2a experiments: 16


,experiment_id,phase,subset_name,window,model_family,run,selection_metric,test_policy
0,phase2a_000,phase_2a_previous_day_priority,daytime_only,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
1,phase2a_001,phase_2a_previous_day_priority,daytime_only,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
2,phase2a_002,phase_2a_previous_day_priority,daytime_only,7,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
3,phase2a_003,phase_2a_previous_day_priority,daytime_only,7,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
4,phase2a_004,phase_2a_previous_day_priority,daytime_only,14,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
5,phase2a_005,phase_2a_previous_day_priority,daytime_only,14,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
6,phase2a_006,phase_2a_previous_day_priority,daytime_only,14,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
7,phase2a_007,phase_2a_previous_day_priority,daytime_only,14,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
8,phase2a_008,phase_2a_previous_day_priority,no_high_sleep_session,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
9,phase2a_009,phase_2a_previous_day_priority,no_high_sleep_session,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인


,subset_name,window,experiments,models
0,daytime_only,7,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
1,daytime_only,14,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
2,no_high_sleep_session,7,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
3,no_high_sleep_session,14,4,"[bilstm, cnn_1d, gru, mlp_current_day]"


In [11]:
# Previous-day Phase 2A 학습 준비 셀
# 목적:
# - previous-day sequence tensor 경로를 사용하도록 loader를 다시 정의한다.
# - Phase 2A experiment grid를 읽는다.
# - 기존 모델 클래스/build_model/train_one_experiment 구조를 재사용한다.
# - 단, load_experiment_data 함수는 previous-day 경로에 맞게 다시 정의한다.

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PREVIOUS_DAY_DIR = PROCESSED_DIR / "deep_learning_previous_day"
PREVIOUS_DAY_SEQUENCE_ROOT = PREVIOUS_DAY_DIR / "sequences_by_subset"

EXPERIMENT_DIR = PREVIOUS_DAY_DIR / "experiments"
PHASE_2A_GRID_PATH = EXPERIMENT_DIR / "phase_2a_previous_day_experiment_grid.csv"

PHASE_2A_OUTPUT_ROOT = EXPERIMENT_DIR / "phase_2a_outputs"
PHASE_2A_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 80
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

phase_2a_grid = pd.read_csv(PHASE_2A_GRID_PATH, encoding="utf-8-sig")

print("PREVIOUS_DAY_SEQUENCE_ROOT:", PREVIOUS_DAY_SEQUENCE_ROOT)
print("PHASE_2A_GRID_PATH:", PHASE_2A_GRID_PATH)
print("PHASE_2A_OUTPUT_ROOT:", PHASE_2A_OUTPUT_ROOT)
print("DEVICE:", DEVICE)
print("phase_2a experiments:", len(phase_2a_grid))

display(phase_2a_grid)


class SleepTensorDataset(Dataset):
    # 하나의 split tensor를 PyTorch Dataset으로 감싸는 클래스
    def __init__(self, arrays, input_key):
        self.X = torch.tensor(arrays[input_key], dtype=torch.float32)
        self.y = torch.tensor(arrays["y"], dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def model_input_key(model_family):
    # 모델 종류에 따라 사용할 입력 tensor를 결정한다.
    if model_family == "mlp_current_day":
        return "X_mlp_current_day"

    if model_family == "mlp_flattened_window":
        return "X_mlp_flattened"

    return "X_sequence"


def load_split_arrays(subset_name, window, split):
    # previous-day subset/window/split에 해당하는 npz tensor를 불러온다.
    path = PREVIOUS_DAY_SEQUENCE_ROOT / subset_name / f"window_{window}" / f"{split}.npz"

    if not path.exists():
        raise FileNotFoundError(path)

    return np.load(path, allow_pickle=True)


def load_experiment_data(subset_name, window, model_family, batch_size=BATCH_SIZE):
    # 하나의 previous-day 실험 조합에 필요한 train/validation/test DataLoader를 생성한다.
    input_key = model_input_key(model_family)

    train_arrays = load_split_arrays(subset_name, window, "train")
    validation_arrays = load_split_arrays(subset_name, window, "validation")
    test_arrays = load_split_arrays(subset_name, window, "test")

    train_dataset = SleepTensorDataset(train_arrays, input_key)
    validation_dataset = SleepTensorDataset(validation_arrays, input_key)
    test_dataset = SleepTensorDataset(test_arrays, input_key)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    input_shape = tuple(train_dataset.X.shape[1:])
    positive_rate = float(train_dataset.y.mean().item())

    return {
        "input_key": input_key,
        "input_shape": input_shape,
        "train_loader": train_loader,
        "validation_loader": validation_loader,
        "test_loader": test_loader,
        "train_arrays": train_arrays,
        "validation_arrays": validation_arrays,
        "test_arrays": test_arrays,
        "positive_rate": positive_rate,
    }


# 대표 조합 하나 로딩 확인
sample_row = phase_2a_grid.iloc[0]

sample_data = load_experiment_data(
    subset_name=sample_row["subset_name"],
    window=int(sample_row["window"]),
    model_family=sample_row["model_family"],
)

print("sample experiment:")
print(sample_row.to_dict())
print("input_key:", sample_data["input_key"])
print("input_shape:", sample_data["input_shape"])
print("train positive_rate:", sample_data["positive_rate"])

batch_X, batch_y = next(iter(sample_data["train_loader"]))
print("batch_X shape:", batch_X.shape)
print("batch_y shape:", batch_y.shape)

PREVIOUS_DAY_SEQUENCE_ROOT: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\sequences_by_subset
PHASE_2A_GRID_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_previous_day_experiment_grid.csv
PHASE_2A_OUTPUT_ROOT: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs
DEVICE: cpu
phase_2a experiments: 16


,experiment_id,phase,subset_name,window,model_family,run,selection_metric,test_policy
0,phase2a_000,phase_2a_previous_day_priority,daytime_only,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
1,phase2a_001,phase_2a_previous_day_priority,daytime_only,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
2,phase2a_002,phase_2a_previous_day_priority,daytime_only,7,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
3,phase2a_003,phase_2a_previous_day_priority,daytime_only,7,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
4,phase2a_004,phase_2a_previous_day_priority,daytime_only,14,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
5,phase2a_005,phase_2a_previous_day_priority,daytime_only,14,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
6,phase2a_006,phase_2a_previous_day_priority,daytime_only,14,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
7,phase2a_007,phase_2a_previous_day_priority,daytime_only,14,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
8,phase2a_008,phase_2a_previous_day_priority,no_high_sleep_session,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
9,phase2a_009,phase_2a_previous_day_priority,no_high_sleep_session,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인


sample experiment:
{'experiment_id': 'phase2a_000', 'phase': 'phase_2a_previous_day_priority', 'subset_name': 'daytime_only', 'window': 7, 'model_family': 'mlp_current_day', 'run': True, 'selection_metric': 'validation_balanced_accuracy', 'test_policy': 'validation ranking 이후 확인'}
input_key: X_mlp_current_day
input_shape: (19,)
train positive_rate: 0.40214887261390686
batch_X shape: torch.Size([32, 19])
batch_y shape: torch.Size([32])


In [13]:
# Previous-day Phase 2A 학습 준비 셀
# 목적:
# - previous-day sequence tensor 경로를 사용하도록 loader를 다시 정의한다.
# - Phase 2A experiment grid를 읽는다.
# - 기존 모델 클래스/build_model/train_one_experiment 구조를 재사용한다.
# - 단, load_experiment_data 함수는 previous-day 경로에 맞게 다시 정의한다.

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PREVIOUS_DAY_DIR = PROCESSED_DIR / "deep_learning_previous_day"
PREVIOUS_DAY_SEQUENCE_ROOT = PREVIOUS_DAY_DIR / "sequences_by_subset"

EXPERIMENT_DIR = PREVIOUS_DAY_DIR / "experiments"
PHASE_2A_GRID_PATH = EXPERIMENT_DIR / "phase_2a_previous_day_experiment_grid.csv"

PHASE_2A_OUTPUT_ROOT = EXPERIMENT_DIR / "phase_2a_outputs"
PHASE_2A_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 80
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

phase_2a_grid = pd.read_csv(PHASE_2A_GRID_PATH, encoding="utf-8-sig")

print("PREVIOUS_DAY_SEQUENCE_ROOT:", PREVIOUS_DAY_SEQUENCE_ROOT)
print("PHASE_2A_GRID_PATH:", PHASE_2A_GRID_PATH)
print("PHASE_2A_OUTPUT_ROOT:", PHASE_2A_OUTPUT_ROOT)
print("DEVICE:", DEVICE)
print("phase_2a experiments:", len(phase_2a_grid))

display(phase_2a_grid)


class SleepTensorDataset(Dataset):
    # 하나의 split tensor를 PyTorch Dataset으로 감싸는 클래스
    def __init__(self, arrays, input_key):
        self.X = torch.tensor(arrays[input_key], dtype=torch.float32)
        self.y = torch.tensor(arrays["y"], dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def model_input_key(model_family):
    # 모델 종류에 따라 사용할 입력 tensor를 결정한다.
    if model_family == "mlp_current_day":
        return "X_mlp_current_day"

    if model_family == "mlp_flattened_window":
        return "X_mlp_flattened"

    return "X_sequence"


def load_split_arrays(subset_name, window, split):
    # previous-day subset/window/split에 해당하는 npz tensor를 불러온다.
    path = PREVIOUS_DAY_SEQUENCE_ROOT / subset_name / f"window_{window}" / f"{split}.npz"

    if not path.exists():
        raise FileNotFoundError(path)

    return np.load(path, allow_pickle=True)


def load_experiment_data(subset_name, window, model_family, batch_size=BATCH_SIZE):
    # 하나의 previous-day 실험 조합에 필요한 train/validation/test DataLoader를 생성한다.
    input_key = model_input_key(model_family)

    train_arrays = load_split_arrays(subset_name, window, "train")
    validation_arrays = load_split_arrays(subset_name, window, "validation")
    test_arrays = load_split_arrays(subset_name, window, "test")

    train_dataset = SleepTensorDataset(train_arrays, input_key)
    validation_dataset = SleepTensorDataset(validation_arrays, input_key)
    test_dataset = SleepTensorDataset(test_arrays, input_key)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    input_shape = tuple(train_dataset.X.shape[1:])
    positive_rate = float(train_dataset.y.mean().item())

    return {
        "input_key": input_key,
        "input_shape": input_shape,
        "train_loader": train_loader,
        "validation_loader": validation_loader,
        "test_loader": test_loader,
        "train_arrays": train_arrays,
        "validation_arrays": validation_arrays,
        "test_arrays": test_arrays,
        "positive_rate": positive_rate,
    }


# 대표 조합 하나 로딩 확인
sample_row = phase_2a_grid.iloc[0]

sample_data = load_experiment_data(
    subset_name=sample_row["subset_name"],
    window=int(sample_row["window"]),
    model_family=sample_row["model_family"],
)

print("sample experiment:")
print(sample_row.to_dict())
print("input_key:", sample_data["input_key"])
print("input_shape:", sample_data["input_shape"])
print("train positive_rate:", sample_data["positive_rate"])

batch_X, batch_y = next(iter(sample_data["train_loader"]))
print("batch_X shape:", batch_X.shape)
print("batch_y shape:", batch_y.shape)

PREVIOUS_DAY_SEQUENCE_ROOT: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\sequences_by_subset
PHASE_2A_GRID_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_previous_day_experiment_grid.csv
PHASE_2A_OUTPUT_ROOT: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs
DEVICE: cpu
phase_2a experiments: 16


,experiment_id,phase,subset_name,window,model_family,run,selection_metric,test_policy
0,phase2a_000,phase_2a_previous_day_priority,daytime_only,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
1,phase2a_001,phase_2a_previous_day_priority,daytime_only,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
2,phase2a_002,phase_2a_previous_day_priority,daytime_only,7,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
3,phase2a_003,phase_2a_previous_day_priority,daytime_only,7,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
4,phase2a_004,phase_2a_previous_day_priority,daytime_only,14,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
5,phase2a_005,phase_2a_previous_day_priority,daytime_only,14,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
6,phase2a_006,phase_2a_previous_day_priority,daytime_only,14,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
7,phase2a_007,phase_2a_previous_day_priority,daytime_only,14,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
8,phase2a_008,phase_2a_previous_day_priority,no_high_sleep_session,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
9,phase2a_009,phase_2a_previous_day_priority,no_high_sleep_session,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인


sample experiment:
{'experiment_id': 'phase2a_000', 'phase': 'phase_2a_previous_day_priority', 'subset_name': 'daytime_only', 'window': 7, 'model_family': 'mlp_current_day', 'run': True, 'selection_metric': 'validation_balanced_accuracy', 'test_policy': 'validation ranking 이후 확인'}
input_key: X_mlp_current_day
input_shape: (19,)
train positive_rate: 0.40214887261390686
batch_X shape: torch.Size([32, 19])
batch_y shape: torch.Size([32])


In [16]:
# Previous-day Phase 2A 모델/학습 함수 정의 셀
# 목적:
# - 새 노트북에서 필요한 모델 클래스와 학습/평가 함수를 모두 정의한다.
# - Phase 2A에서는 mlp_current_day, GRU, BiLSTM, 1D CNN을 사용한다.

from pathlib import Path
import random
import json
from copy import deepcopy

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

print("torch:", torch.__version__)
print("nn loaded:", nn is not None)


class MLPCurrentDay(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.30):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class GRUClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, dropout=0.20):
        super().__init__()

        effective_dropout = dropout if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=effective_dropout,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        output, hidden = self.gru(x)
        last_hidden = hidden[-1]
        return self.head(last_hidden).squeeze(-1)


class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=48, num_layers=1, dropout=0.20):
        super().__init__()

        effective_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=effective_dropout,
            bidirectional=True,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 1),
        )

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)

        forward_last = hidden[-2]
        backward_last = hidden[-1]
        combined = torch.cat([forward_last, backward_last], dim=1)

        return self.head(combined).squeeze(-1)


class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim, channels=64, dropout=0.25):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
        )

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(channels, 1),
        )

    def forward(self, x):
        # 입력: batch x time x features
        # Conv1d 입력: batch x features x time
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x).squeeze(-1)


def build_model(model_family, input_shape):
    # input_shape:
    # - MLP: (features,)
    # - sequence model: (window, features)

    if model_family == "mlp_current_day":
        input_dim = input_shape[0]
        return MLPCurrentDay(input_dim=input_dim)

    if model_family == "gru":
        input_dim = input_shape[-1]
        return GRUClassifier(input_dim=input_dim)

    if model_family == "bilstm":
        input_dim = input_shape[-1]
        return BiLSTMClassifier(input_dim=input_dim)

    if model_family == "cnn_1d":
        input_dim = input_shape[-1]
        return CNN1DClassifier(input_dim=input_dim)

    raise ValueError(f"지원하지 않는 model_family입니다: {model_family}")


def make_pos_weight(train_arrays):
    # BCEWithLogitsLoss의 class imbalance 보정값을 만든다.
    y = train_arrays["y"].astype(int)
    positives = int((y == 1).sum())
    negatives = int((y == 0).sum())

    if positives == 0:
        return torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

    return torch.tensor(negatives / positives, dtype=torch.float32, device=DEVICE)


def predict_logits(model, loader):
    model.eval()

    all_logits = []
    all_y = []

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(DEVICE)
            logits = model(batch_X)

            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(batch_y.detach().cpu().numpy())

    logits = np.concatenate(all_logits)
    y_true = np.concatenate(all_y).astype(int)

    return logits, y_true


def metrics_from_logits(logits, y_true, threshold=0.5):
    probabilities = 1.0 / (1.0 + np.exp(-logits))
    y_pred = (probabilities >= threshold).astype(int)

    result = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "positive_prediction_rate": float(y_pred.mean()),
        "target_positive_rate": float(y_true.mean()),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, probabilities)
        result["average_precision"] = average_precision_score(y_true, probabilities)
    else:
        result["roc_auc"] = np.nan
        result["average_precision"] = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result.update({
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    })

    return result, probabilities, y_pred


def train_one_experiment(row, verbose=True):
    experiment_id = row["experiment_id"]
    subset_name = row["subset_name"]
    window = int(row["window"])
    model_family = row["model_family"]

    data = load_experiment_data(
        subset_name=subset_name,
        window=window,
        model_family=model_family,
    )

    model = build_model(model_family, data["input_shape"]).to(DEVICE)

    pos_weight = make_pos_weight(data["train_arrays"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_validation_balanced_accuracy = -np.inf
    best_epoch = -1
    patience_counter = 0

    history_rows = []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_losses = []

        for batch_X, batch_y in data["train_loader"]:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            train_losses.append(float(loss.item()))

        validation_logits, validation_y = predict_logits(model, data["validation_loader"])
        validation_metrics, _, _ = metrics_from_logits(validation_logits, validation_y)

        train_loss = float(np.mean(train_losses))

        history_rows.append({
            "experiment_id": experiment_id,
            "subset_name": subset_name,
            "window": window,
            "model_family": model_family,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_balanced_accuracy": validation_metrics["balanced_accuracy"],
            "validation_roc_auc": validation_metrics["roc_auc"],
            "validation_f1": validation_metrics["f1"],
            "validation_recall": validation_metrics["recall"],
            "validation_precision": validation_metrics["precision"],
        })

        current_score = validation_metrics["balanced_accuracy"]

        if current_score > best_validation_balanced_accuracy:
            best_validation_balanced_accuracy = current_score
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(
                f"{experiment_id} epoch={epoch:03d} "
                f"loss={train_loss:.4f} "
                f"val_bal_acc={current_score:.4f}"
            )

        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)

    metric_rows = []
    prediction_rows = []

    for split_name, loader in [
        ("validation", data["validation_loader"]),
        ("test", data["test_loader"]),
    ]:
        logits, y_true = predict_logits(model, loader)
        metrics, probabilities, y_pred = metrics_from_logits(logits, y_true)

        metric_row = {
            "experiment_id": experiment_id,
            "subset_name": subset_name,
            "window": window,
            "model_family": model_family,
            "split": split_name,
            "best_epoch": best_epoch,
            "input_key": data["input_key"],
            "input_shape": str(data["input_shape"]),
        }
        metric_row.update(metrics)
        metric_rows.append(metric_row)

        arrays = data[f"{split_name}_arrays"]

        split_predictions = pd.DataFrame({
            "experiment_id": experiment_id,
            "subset_name": subset_name,
            "window": window,
            "model_family": model_family,
            "split": split_name,
            "participant_object_id": arrays["participant_object_id"],
            "window_start_date": arrays["window_start_date"],
            "window_end_date": arrays["window_end_date"],
            "feature_start_date": arrays["feature_start_date"],
            "feature_end_date": arrays["feature_end_date"],
            "y_true": y_true,
            "probability": probabilities,
            "y_pred": y_pred,
        })

        prediction_rows.append(split_predictions)

    metrics_df = pd.DataFrame(metric_rows)
    history_df = pd.DataFrame(history_rows)
    predictions_df = pd.concat(prediction_rows, ignore_index=True)

    return metrics_df, history_df, predictions_df

torch: 2.12.1+cpu
nn loaded: True


In [17]:
# Previous-day Phase 2A smoke training 셀
# 목적:
# - previous-day 데이터에서 학습 함수가 정상 작동하는지 1개 실험만 확인한다.
# - 전체 16개 실행 전에 metrics/history/predictions 생성 여부를 확인한다.

ORIGINAL_NUM_EPOCHS = NUM_EPOCHS
ORIGINAL_PATIENCE = PATIENCE

NUM_EPOCHS = 5
PATIENCE = 3

smoke_row = phase_2a_grid.iloc[0].copy()

print("smoke experiment:")
print(smoke_row.to_dict())

smoke_metrics, smoke_history, smoke_predictions = train_one_experiment(
    smoke_row,
    verbose=True,
)

NUM_EPOCHS = ORIGINAL_NUM_EPOCHS
PATIENCE = ORIGINAL_PATIENCE

print("\nsmoke_metrics:")
display(smoke_metrics)

print("\nsmoke_history tail:")
display(smoke_history.tail())

print("\nsmoke_predictions head:")
display(smoke_predictions.head())

print("metrics shape:", smoke_metrics.shape)
print("history shape:", smoke_history.shape)
print("predictions shape:", smoke_predictions.shape)

smoke experiment:
{'experiment_id': 'phase2a_000', 'phase': 'phase_2a_previous_day_priority', 'subset_name': 'daytime_only', 'window': 7, 'model_family': 'mlp_current_day', 'run': True, 'selection_metric': 'validation_balanced_accuracy', 'test_policy': 'validation ranking 이후 확인'}
phase2a_000 epoch=001 loss=0.8351 val_bal_acc=0.6223

smoke_metrics:


,experiment_id,subset_name,window,model_family,split,best_epoch,input_key,input_shape,threshold,accuracy,...,precision,recall,positive_prediction_rate,target_positive_rate,roc_auc,average_precision,tn,fp,fn,tp
0,phase2a_000,daytime_only,7,mlp_current_day,validation,1,X_mlp_current_day,"(19,)",0.5,0.595918,...,0.475177,0.728261,0.57551,0.375510,0.632495,0.448672,79,74,25,67
1,phase2a_000,daytime_only,7,mlp_current_day,test,1,X_mlp_current_day,"(19,)",0.5,0.597973,...,0.512987,0.642276,0.52027,0.415541,0.634757,0.526342,98,75,44,79



smoke_history tail:


,experiment_id,subset_name,window,model_family,epoch,train_loss,validation_balanced_accuracy,validation_roc_auc,validation_f1,validation_recall,validation_precision
0,phase2a_000,daytime_only,7,mlp_current_day,1,0.835146,0.622300,0.632495,0.575107,0.728261,0.475177
1,phase2a_000,daytime_only,7,mlp_current_day,2,0.789649,0.609228,0.618073,0.565401,0.728261,0.462069
2,phase2a_000,daytime_only,7,mlp_current_day,3,0.782871,0.613669,0.630790,0.552995,0.652174,0.480000
3,phase2a_000,daytime_only,7,mlp_current_day,4,0.773930,0.581024,0.603581,0.520548,0.619565,0.448819



smoke_predictions head:


,experiment_id,subset_name,window,model_family,split,participant_object_id,window_start_date,window_end_date,feature_start_date,feature_end_date,y_true,probability,y_pred
0,phase2a_000,daytime_only,7,mlp_current_day,validation,621e2f3967b776a240c654db,2021-05-26,2021-06-01,2021-05-25,2021-05-31,0,0.617326,1
1,phase2a_000,daytime_only,7,mlp_current_day,validation,621e2f3967b776a240c654db,2021-05-27,2021-06-02,2021-05-26,2021-06-01,0,0.635195,1
2,phase2a_000,daytime_only,7,mlp_current_day,validation,621e2f3967b776a240c654db,2021-05-28,2021-06-03,2021-05-27,2021-06-02,1,0.615015,1
3,phase2a_000,daytime_only,7,mlp_current_day,validation,621e2f3967b776a240c654db,2021-05-29,2021-06-04,2021-05-28,2021-06-03,0,0.670162,1
4,phase2a_000,daytime_only,7,mlp_current_day,validation,621e2f3967b776a240c654db,2021-05-30,2021-06-05,2021-05-29,2021-06-04,1,0.641052,1


metrics shape: (2, 22)
history shape: (4, 11)
predictions shape: (541, 13)


In [19]:
# Previous-day Phase 2A 전체 학습 실행 셀
# 목적:
# - previous-day Phase 2A 16개 실험을 실행한다.
# - metrics/history/predictions를 누적 저장한다.
# - 중간에 멈춰도 이미 완료된 experiment_id는 건너뛴다.

PHASE_2A_METRICS_PATH = PHASE_2A_OUTPUT_ROOT / "phase_2a_previous_day_model_metrics.csv"
PHASE_2A_HISTORY_PATH = PHASE_2A_OUTPUT_ROOT / "phase_2a_previous_day_training_history.csv"
PHASE_2A_PREDICTIONS_PATH = PHASE_2A_OUTPUT_ROOT / "phase_2a_previous_day_model_predictions.csv"

RUN_PHASE_2A_FULL = True

print("PHASE_2A_METRICS_PATH:", PHASE_2A_METRICS_PATH)
print("PHASE_2A_HISTORY_PATH:", PHASE_2A_HISTORY_PATH)
print("PHASE_2A_PREDICTIONS_PATH:", PHASE_2A_PREDICTIONS_PATH)
print("RUN_PHASE_2A_FULL:", RUN_PHASE_2A_FULL)

if RUN_PHASE_2A_FULL:
    completed_experiment_ids = set()

    if PHASE_2A_METRICS_PATH.exists():
        existing_metrics = pd.read_csv(PHASE_2A_METRICS_PATH, encoding="utf-8-sig")
        completed_experiment_ids = set(existing_metrics["experiment_id"].unique())
        print("completed experiments:", len(completed_experiment_ids))

    metrics_outputs = []
    history_outputs = []
    prediction_outputs = []

    for idx, row in phase_2a_grid.iterrows():
        experiment_id = row["experiment_id"]

        if experiment_id in completed_experiment_ids:
            print(f"[SKIP] {experiment_id} already completed")
            continue

        print("\n" + "=" * 100)
        print(
            f"[RUN] {idx + 1}/{len(phase_2a_grid)} "
            f"{experiment_id} | {row['subset_name']} | "
            f"window={row['window']} | model={row['model_family']}"
        )

        metrics_df, history_df, predictions_df = train_one_experiment(
            row,
            verbose=True,
        )

        metrics_outputs.append(metrics_df)
        history_outputs.append(history_df)
        prediction_outputs.append(predictions_df)

        if metrics_outputs:
            new_metrics = pd.concat(metrics_outputs, ignore_index=True)

            if PHASE_2A_METRICS_PATH.exists():
                old_metrics = pd.read_csv(PHASE_2A_METRICS_PATH, encoding="utf-8-sig")
                save_metrics = pd.concat([old_metrics, new_metrics], ignore_index=True)
                save_metrics = save_metrics.drop_duplicates(
                    subset=["experiment_id", "split"],
                    keep="last",
                )
            else:
                save_metrics = new_metrics

            save_metrics.to_csv(
                PHASE_2A_METRICS_PATH,
                index=False,
                encoding="utf-8-sig",
            )

        if history_outputs:
            new_history = pd.concat(history_outputs, ignore_index=True)

            if PHASE_2A_HISTORY_PATH.exists():
                old_history = pd.read_csv(PHASE_2A_HISTORY_PATH, encoding="utf-8-sig")
                save_history = pd.concat([old_history, new_history], ignore_index=True)
                save_history = save_history.drop_duplicates(
                    subset=["experiment_id", "epoch"],
                    keep="last",
                )
            else:
                save_history = new_history

            save_history.to_csv(
                PHASE_2A_HISTORY_PATH,
                index=False,
                encoding="utf-8-sig",
            )

        if prediction_outputs:
            new_predictions = pd.concat(prediction_outputs, ignore_index=True)

            if PHASE_2A_PREDICTIONS_PATH.exists():
                old_predictions = pd.read_csv(PHASE_2A_PREDICTIONS_PATH, encoding="utf-8-sig")
                save_predictions = pd.concat(
                    [old_predictions, new_predictions],
                    ignore_index=True,
                )
                save_predictions = save_predictions.drop_duplicates(
                    subset=[
                        "experiment_id",
                        "split",
                        "participant_object_id",
                        "window_start_date",
                        "window_end_date",
                    ],
                    keep="last",
                )
            else:
                save_predictions = new_predictions

            save_predictions.to_csv(
                PHASE_2A_PREDICTIONS_PATH,
                index=False,
                encoding="utf-8-sig",
            )

        print(f"[SAVED] {experiment_id}")

    print("\nPrevious-day Phase 2A 학습 완료")
else:
    print("RUN_PHASE_2A_FULL이 False입니다.")
    print("전체 16개 실험을 실행하려면 RUN_PHASE_2A_FULL = True로 바꾸고 이 셀을 다시 실행하세요.")

PHASE_2A_METRICS_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs\phase_2a_previous_day_model_metrics.csv
PHASE_2A_HISTORY_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs\phase_2a_previous_day_training_history.csv
PHASE_2A_PREDICTIONS_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs\phase_2a_previous_day_model_predictions.csv
RUN_PHASE_2A_FULL: True

[RUN] 1/16 phase2a_000 | daytime_only | window=7 | model=mlp_current_day
phase2a_000 epoch=001 loss=0.8626 val_bal_acc=0.6050
[SAVED] phase2a_000

[RUN] 2/16 phase2a_001 | daytime_only | window=7 | model=gru
phase2a_001 epoch=001 loss=0.7983 val_bal_acc=0.5985
[SAVED] phase2a_001

[RUN] 3/16 phase2a_002 | daytime_only | window=7 | model=bilstm
phase2a_002 epoch=001 loss=0.7930 val_bal_acc=0.6528
[SAVED] phase2a_002

[RUN] 4/16 phase2a_003 | daytime_only | window=7 

In [20]:
# Previous-day Phase 2A 결과 확인 셀
# 목적:
# - previous-day Phase 2A 결과를 validation balanced accuracy 기준으로 정렬한다.
# - validation 기준 선택 이후 test metric을 붙여서 일반화 gap을 확인한다.
# - same-date Phase 1A 결과와 비교할 준비를 한다.

import pandas as pd
import numpy as np

phase2_metrics = pd.read_csv(PHASE_2A_METRICS_PATH, encoding="utf-8-sig")
phase2_history = pd.read_csv(PHASE_2A_HISTORY_PATH, encoding="utf-8-sig")
phase2_predictions = pd.read_csv(PHASE_2A_PREDICTIONS_PATH, encoding="utf-8-sig")

print("metrics shape:", phase2_metrics.shape)
print("history shape:", phase2_history.shape)
print("predictions shape:", phase2_predictions.shape)
print("experiments in metrics:", phase2_metrics["experiment_id"].nunique())
print(phase2_metrics["split"].value_counts())

validation_metrics = phase2_metrics[phase2_metrics["split"] == "validation"].copy()
test_metrics = phase2_metrics[phase2_metrics["split"] == "test"].copy()

validation_rank = validation_metrics.sort_values(
    ["balanced_accuracy", "roc_auc", "average_precision"],
    ascending=False,
).reset_index(drop=True)

print("\nValidation ranking:")
display(validation_rank)

paired_results = validation_metrics.merge(
    test_metrics,
    on=["experiment_id", "subset_name", "window", "model_family"],
    suffixes=("_validation", "_test"),
)

paired_results["balanced_accuracy_gap"] = (
    paired_results["balanced_accuracy_validation"]
    - paired_results["balanced_accuracy_test"]
)

paired_results["roc_auc_gap"] = (
    paired_results["roc_auc_validation"]
    - paired_results["roc_auc_test"]
)

paired_rank = paired_results.sort_values(
    ["balanced_accuracy_validation", "roc_auc_validation", "average_precision_validation"],
    ascending=False,
).reset_index(drop=True)

print("\nPaired validation/test ranking:")
display(
    paired_rank[
        [
            "experiment_id",
            "subset_name",
            "window",
            "model_family",
            "best_epoch_validation",
            "balanced_accuracy_validation",
            "roc_auc_validation",
            "average_precision_validation",
            "f1_validation",
            "balanced_accuracy_test",
            "roc_auc_test",
            "average_precision_test",
            "f1_test",
            "balanced_accuracy_gap",
            "roc_auc_gap",
        ]
    ]
)

subset_summary = (
    paired_results
    .groupby("subset_name", as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_val_bal_acc=("balanced_accuracy_validation", "mean"),
        max_val_bal_acc=("balanced_accuracy_validation", "max"),
        mean_test_bal_acc=("balanced_accuracy_test", "mean"),
        max_test_bal_acc=("balanced_accuracy_test", "max"),
        mean_bal_acc_gap=("balanced_accuracy_gap", "mean"),
        median_bal_acc_gap=("balanced_accuracy_gap", "median"),
    )
    .sort_values("max_val_bal_acc", ascending=False)
)

print("\nSubset summary:")
display(subset_summary)

model_summary = (
    paired_results
    .groupby("model_family", as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_val_bal_acc=("balanced_accuracy_validation", "mean"),
        max_val_bal_acc=("balanced_accuracy_validation", "max"),
        mean_test_bal_acc=("balanced_accuracy_test", "mean"),
        max_test_bal_acc=("balanced_accuracy_test", "max"),
        mean_bal_acc_gap=("balanced_accuracy_gap", "mean"),
    )
    .sort_values("max_val_bal_acc", ascending=False)
)

print("\nModel summary:")
display(model_summary)

window_summary = (
    paired_results
    .groupby("window", as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_val_bal_acc=("balanced_accuracy_validation", "mean"),
        max_val_bal_acc=("balanced_accuracy_validation", "max"),
        mean_test_bal_acc=("balanced_accuracy_test", "mean"),
        max_test_bal_acc=("balanced_accuracy_test", "max"),
        mean_bal_acc_gap=("balanced_accuracy_gap", "mean"),
    )
    .sort_values("window")
)

print("\nWindow summary:")
display(window_summary)

metrics shape: (32, 22)
history shape: (75, 11)
predictions shape: (7064, 13)
experiments in metrics: 16
split
validation    16
test          16
Name: count, dtype: int64

Validation ranking:


,experiment_id,subset_name,window,model_family,split,best_epoch,input_key,input_shape,threshold,accuracy,...,precision,recall,positive_prediction_rate,target_positive_rate,roc_auc,average_precision,tn,fp,fn,tp
0,phase2a_015,no_high_sleep_session,14,cnn_1d,validation,3,X_sequence,"(14, 127)",0.5,0.720588,...,0.575758,0.791667,0.485294,0.352941,0.780777,0.585075,60,28,10,38
1,phase2a_012,no_high_sleep_session,14,mlp_current_day,validation,3,X_mlp_current_day,"(127,)",0.5,0.705882,...,0.562500,0.750000,0.470588,0.352941,0.769413,0.554165,60,28,12,36
2,phase2a_014,no_high_sleep_session,14,bilstm,validation,3,X_sequence,"(14, 127)",0.5,0.705882,...,0.564516,0.729167,0.455882,0.352941,0.774621,0.583874,61,27,13,35
3,phase2a_007,daytime_only,14,cnn_1d,validation,2,X_sequence,"(14, 19)",0.5,0.742647,...,0.644444,0.604167,0.330882,0.352941,0.759943,0.567897,72,16,19,29
4,phase2a_013,no_high_sleep_session,14,gru,validation,2,X_sequence,"(14, 127)",0.5,0.713235,...,0.581818,0.666667,0.404412,0.352941,0.777225,0.581356,65,23,16,32
5,phase2a_006,daytime_only,14,bilstm,validation,1,X_sequence,"(14, 19)",0.5,0.691176,...,0.546875,0.729167,0.470588,0.352941,0.706203,0.534636,59,29,13,35
6,phase2a_010,no_high_sleep_session,7,bilstm,validation,2,X_sequence,"(7, 127)",0.5,0.636735,...,0.510638,0.782609,0.575510,0.375510,0.695510,0.549066,84,69,20,72
7,phase2a_005,daytime_only,14,gru,validation,5,X_sequence,"(14, 19)",0.5,0.698529,...,0.577778,0.541667,0.330882,0.352941,0.771307,0.554877,69,19,22,26
8,phase2a_011,no_high_sleep_session,7,cnn_1d,validation,4,X_sequence,"(7, 127)",0.5,0.628571,...,0.503448,0.793478,0.591837,0.375510,0.690111,0.499646,81,72,19,73
9,phase2a_002,daytime_only,7,bilstm,validation,1,X_sequence,"(7, 19)",0.5,0.644898,...,0.520661,0.684783,0.493878,0.375510,0.689400,0.521805,95,58,29,63



Paired validation/test ranking:


,experiment_id,subset_name,window,model_family,best_epoch_validation,balanced_accuracy_validation,roc_auc_validation,average_precision_validation,f1_validation,balanced_accuracy_test,roc_auc_test,average_precision_test,f1_test,balanced_accuracy_gap,roc_auc_gap
0,phase2a_015,no_high_sleep_session,14,cnn_1d,3,0.736742,0.780777,0.585075,0.666667,0.444765,0.444056,0.450391,0.361582,0.291977,0.336721
1,phase2a_012,no_high_sleep_session,14,mlp_current_day,3,0.715909,0.769413,0.554165,0.642857,0.467086,0.440745,0.456275,0.417112,0.248823,0.328668
2,phase2a_014,no_high_sleep_session,14,bilstm,3,0.711174,0.774621,0.583874,0.636364,0.472335,0.438948,0.439555,0.311688,0.238839,0.335673
3,phase2a_007,daytime_only,14,cnn_1d,2,0.711174,0.759943,0.567897,0.623656,0.567436,0.554053,0.556660,0.466258,0.143738,0.205890
4,phase2a_013,no_high_sleep_session,14,gru,2,0.702652,0.777225,0.581356,0.621359,0.455642,0.433084,0.450329,0.386740,0.247010,0.344141
5,phase2a_006,daytime_only,14,bilstm,1,0.699811,0.706203,0.534636,0.625000,0.586825,0.606261,0.578799,0.568528,0.112986,0.099941
6,phase2a_010,no_high_sleep_session,7,bilstm,2,0.665814,0.695510,0.549066,0.618026,0.539241,0.566051,0.438108,0.465863,0.126574,0.129459
7,phase2a_005,daytime_only,14,gru,5,0.662879,0.771307,0.554877,0.559140,0.529604,0.564930,0.526578,0.431138,0.133275,0.206377
8,phase2a_011,no_high_sleep_session,7,cnn_1d,4,0.661445,0.690111,0.499646,0.616034,0.479792,0.484045,0.417105,0.309179,0.181653,0.206066
9,phase2a_002,daytime_only,7,bilstm,1,0.652849,0.689400,0.521805,0.591549,0.577988,0.588045,0.522884,0.495798,0.074861,0.101356



Subset summary:


,subset_name,experiments,mean_val_bal_acc,max_val_bal_acc,mean_test_bal_acc,max_test_bal_acc,mean_bal_acc_gap,median_bal_acc_gap
1,no_high_sleep_session,8,0.682128,0.736742,0.492832,0.544199,0.189296,0.210246
0,daytime_only,8,0.643790,0.711174,0.569919,0.617393,0.073872,0.069908



Model summary:


,model_family,experiments,mean_val_bal_acc,max_val_bal_acc,mean_test_bal_acc,max_test_bal_acc,mean_bal_acc_gap
1,cnn_1d,4,0.679958,0.736742,0.527346,0.617393,0.152612
3,mlp_current_day,4,0.642031,0.715909,0.536933,0.592533,0.105098
0,bilstm,4,0.682412,0.711174,0.544097,0.586825,0.138315
2,gru,4,0.647435,0.702652,0.517124,0.544199,0.130311



Window summary:


,window,experiments,mean_val_bal_acc,max_val_bal_acc,mean_test_bal_acc,max_test_bal_acc,mean_bal_acc_gap
0,7,8,0.632855,0.665814,0.553724,0.617393,0.079131
1,14,8,0.693063,0.736742,0.509027,0.586825,0.184037


In [21]:
# Previous-day Phase 2A validation threshold tuning
# 목적:
# - previous-day 각 experiment_id별로 validation에서 balanced accuracy가 최대인 threshold를 찾는다.
# - 찾은 threshold를 test에 그대로 적용한다.
# - test threshold는 직접 최적화하지 않는다.

from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

threshold_grid = np.round(np.arange(0.10, 0.91, 0.01), 2)

def evaluate_with_threshold(y_true, probability, threshold):
    y_pred = (probability >= threshold).astype(int)

    result = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "positive_prediction_rate": float(y_pred.mean()),
        "target_positive_rate": float(y_true.mean()),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, probability)
        result["average_precision"] = average_precision_score(y_true, probability)
    else:
        result["roc_auc"] = np.nan
        result["average_precision"] = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result.update({"tn": tn, "fp": fp, "fn": fn, "tp": tp})

    return result


rows = []

for experiment_id, group in phase2_predictions.groupby("experiment_id"):
    validation_group = group[group["split"] == "validation"].copy()
    test_group = group[group["split"] == "test"].copy()

    validation_scores = []

    for threshold in threshold_grid:
        metric = evaluate_with_threshold(
            validation_group["y_true"].to_numpy(),
            validation_group["probability"].to_numpy(),
            threshold,
        )
        validation_scores.append(metric)

    validation_threshold_df = pd.DataFrame(validation_scores)

    best_validation_row = validation_threshold_df.sort_values(
        ["balanced_accuracy", "f1", "roc_auc"],
        ascending=False,
    ).iloc[0]

    best_threshold = float(best_validation_row["threshold"])

    for split_name, split_group in [
        ("validation", validation_group),
        ("test", test_group),
    ]:
        metric = evaluate_with_threshold(
            split_group["y_true"].to_numpy(),
            split_group["probability"].to_numpy(),
            best_threshold,
        )

        meta = split_group.iloc[0][
            ["experiment_id", "subset_name", "window", "model_family"]
        ].to_dict()

        rows.append({
            **meta,
            "split": split_name,
            "selected_threshold_from_validation": best_threshold,
            **metric,
        })

phase2_threshold_metrics = pd.DataFrame(rows)

validation_threshold_metrics = phase2_threshold_metrics[
    phase2_threshold_metrics["split"] == "validation"
].copy()

test_threshold_metrics = phase2_threshold_metrics[
    phase2_threshold_metrics["split"] == "test"
].copy()

paired_threshold_results = validation_threshold_metrics.merge(
    test_threshold_metrics,
    on=["experiment_id", "subset_name", "window", "model_family"],
    suffixes=("_validation", "_test"),
)

paired_threshold_results["balanced_accuracy_gap"] = (
    paired_threshold_results["balanced_accuracy_validation"]
    - paired_threshold_results["balanced_accuracy_test"]
)

paired_threshold_rank = paired_threshold_results.sort_values(
    ["balanced_accuracy_validation", "roc_auc_validation", "average_precision_validation"],
    ascending=False,
).reset_index(drop=True)

display(
    paired_threshold_rank[
        [
            "experiment_id",
            "subset_name",
            "window",
            "model_family",
            "selected_threshold_from_validation_validation",
            "balanced_accuracy_validation",
            "roc_auc_validation",
            "average_precision_validation",
            "f1_validation",
            "balanced_accuracy_test",
            "roc_auc_test",
            "average_precision_test",
            "f1_test",
            "balanced_accuracy_gap",
        ]
    ]
)

threshold_subset_summary = (
    paired_threshold_results
    .groupby("subset_name", as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_val_bal_acc=("balanced_accuracy_validation", "mean"),
        max_val_bal_acc=("balanced_accuracy_validation", "max"),
        mean_test_bal_acc=("balanced_accuracy_test", "mean"),
        max_test_bal_acc=("balanced_accuracy_test", "max"),
        mean_bal_acc_gap=("balanced_accuracy_gap", "mean"),
        median_bal_acc_gap=("balanced_accuracy_gap", "median"),
    )
    .sort_values("max_val_bal_acc", ascending=False)
)

display(threshold_subset_summary)

phase2_threshold_metrics_path = PHASE_2A_OUTPUT_ROOT / "phase_2a_previous_day_threshold_tuned_metrics.csv"
phase2_threshold_metrics.to_csv(
    phase2_threshold_metrics_path,
    index=False,
    encoding="utf-8-sig",
)

print("saved:", phase2_threshold_metrics_path)

,experiment_id,subset_name,window,model_family,selected_threshold_from_validation_validation,balanced_accuracy_validation,roc_auc_validation,average_precision_validation,f1_validation,balanced_accuracy_test,roc_auc_test,average_precision_test,f1_test,balanced_accuracy_gap
0,phase2a_005,daytime_only,14,gru,0.28,0.781250,0.771307,0.554877,0.714286,0.511019,0.564930,0.526578,0.593750,0.270231
1,phase2a_015,no_high_sleep_session,14,cnn_1d,0.49,0.762311,0.780777,0.585075,0.694915,0.440178,0.444056,0.450391,0.359551,0.322133
2,phase2a_007,daytime_only,14,cnn_1d,0.35,0.755682,0.759943,0.567897,0.688525,0.524260,0.554053,0.556660,0.572650,0.231422
3,phase2a_013,no_high_sleep_session,14,gru,0.34,0.750947,0.777225,0.581356,0.686131,0.408493,0.433084,0.450329,0.478992,0.342454
4,phase2a_012,no_high_sleep_session,14,mlp_current_day,0.42,0.747159,0.769413,0.554165,0.681818,0.424572,0.440745,0.456275,0.469027,0.322587
5,phase2a_014,no_high_sleep_session,14,bilstm,0.32,0.745265,0.774621,0.583874,0.681159,0.454932,0.438948,0.439555,0.504348,0.290333
6,phase2a_006,daytime_only,14,bilstm,0.51,0.717803,0.706203,0.534636,0.641509,0.609761,0.606261,0.578799,0.583333,0.108042
7,phase2a_001,daytime_only,7,gru,0.35,0.685351,0.680378,0.500064,0.641975,0.579092,0.607500,0.505873,0.570492,0.106259
8,phase2a_011,no_high_sleep_session,7,cnn_1d,0.49,0.676648,0.690111,0.499646,0.633745,0.494337,0.484045,0.417105,0.345794,0.182311
9,phase2a_010,no_high_sleep_session,7,bilstm,0.49,0.670148,0.695510,0.549066,0.624473,0.548005,0.566051,0.438108,0.488372,0.122143


,subset_name,experiments,mean_val_bal_acc,max_val_bal_acc,mean_test_bal_acc,max_test_bal_acc,mean_bal_acc_gap,median_bal_acc_gap
0,daytime_only,8,0.686530,0.781250,0.557701,0.609761,0.128829,0.098966
1,no_high_sleep_session,8,0.710482,0.762311,0.477177,0.548640,0.233305,0.236322


saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs\phase_2a_previous_day_threshold_tuned_metrics.csv


In [22]:
# Same-date Phase 1A vs Previous-day Phase 2A 비교 요약
# 목적:
# - same-date subset ablation 결과와 previous-day 결과를 한 표로 비교한다.
# - 시간 기준을 엄격히 했을 때 성능이 얼마나 떨어지는지 정량화한다.

PHASE_1A_THRESHOLD_PATH = (
    PROCESSED_DIR
    / "deep_learning_subset_experiments"
    / "phase_1a_outputs"
    / "phase_1a_threshold_tuned_metrics.csv"
)

PHASE_2A_THRESHOLD_PATH = (
    PHASE_2A_OUTPUT_ROOT
    / "phase_2a_previous_day_threshold_tuned_metrics.csv"
)

phase1_threshold = pd.read_csv(PHASE_1A_THRESHOLD_PATH, encoding="utf-8-sig")
phase2_threshold = pd.read_csv(PHASE_2A_THRESHOLD_PATH, encoding="utf-8-sig")

phase1_threshold["feature_timing"] = "same_date"
phase2_threshold["feature_timing"] = "previous_day"

combined_threshold = pd.concat(
    [phase1_threshold, phase2_threshold],
    ignore_index=True,
)

comparison_summary = (
    combined_threshold
    .groupby(["feature_timing", "subset_name", "split"], as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        max_balanced_accuracy=("balanced_accuracy", "max"),
        mean_roc_auc=("roc_auc", "mean"),
        max_roc_auc=("roc_auc", "max"),
        mean_f1=("f1", "mean"),
        max_f1=("f1", "max"),
    )
    .sort_values(["feature_timing", "subset_name", "split"])
)

display(comparison_summary)

test_comparison = comparison_summary[
    comparison_summary["split"] == "test"
].copy()

display(
    test_comparison.sort_values(
        ["max_balanced_accuracy", "mean_balanced_accuracy"],
        ascending=False,
    )
)

top_candidate_comparison = combined_threshold[
    combined_threshold["split"] == "test"
].sort_values(
    ["balanced_accuracy", "roc_auc", "f1"],
    ascending=False,
).head(20)

display(
    top_candidate_comparison[
        [
            "feature_timing",
            "experiment_id",
            "subset_name",
            "window",
            "model_family",
            "selected_threshold_from_validation",
            "balanced_accuracy",
            "roc_auc",
            "average_precision",
            "f1",
            "precision",
            "recall",
        ]
    ]
)

comparison_summary_path = PHASE_2A_OUTPUT_ROOT / "same_date_vs_previous_day_threshold_comparison.csv"
comparison_summary.to_csv(comparison_summary_path, index=False, encoding="utf-8-sig")

top_candidate_comparison_path = PHASE_2A_OUTPUT_ROOT / "same_date_vs_previous_day_top_test_candidates.csv"
top_candidate_comparison.to_csv(top_candidate_comparison_path, index=False, encoding="utf-8-sig")

print("saved:", comparison_summary_path)
print("saved:", top_candidate_comparison_path)

,feature_timing,subset_name,split,experiments,mean_balanced_accuracy,max_balanced_accuracy,mean_roc_auc,max_roc_auc,mean_f1,max_f1
0,previous_day,daytime_only,test,8,0.557701,0.609761,0.588883,0.628648,0.568540,0.598425
1,previous_day,daytime_only,validation,8,0.686530,0.781250,0.691858,0.771307,0.634719,0.714286
2,previous_day,no_high_sleep_session,test,8,0.477177,0.548640,0.487092,0.566051,0.459691,0.533333
3,previous_day,no_high_sleep_session,validation,8,0.710482,0.762311,0.730732,0.780777,0.658185,0.694915
4,same_date,daytime_only,test,12,0.763507,0.843989,0.844957,0.904679,0.785883,0.852941
5,same_date,daytime_only,validation,12,0.881946,0.952381,0.900994,0.979853,0.821061,0.866667
6,same_date,full_current,test,12,0.594129,0.729021,0.652203,0.811400,0.528818,0.693141
7,same_date,full_current,validation,12,0.831944,0.928571,0.871265,0.950549,0.750658,0.826923
8,same_date,no_high_sleep_session,test,12,0.667121,0.814567,0.725264,0.878255,0.681697,0.829268
9,same_date,no_high_sleep_session,validation,12,0.858180,0.928571,0.883832,0.930403,0.779723,0.838095


,feature_timing,subset_name,split,experiments,mean_balanced_accuracy,max_balanced_accuracy,mean_roc_auc,max_roc_auc,mean_f1,max_f1
4,same_date,daytime_only,test,12,0.763507,0.843989,0.844957,0.904679,0.785883,0.852941
8,same_date,no_high_sleep_session,test,12,0.667121,0.814567,0.725264,0.878255,0.681697,0.829268
6,same_date,full_current,test,12,0.594129,0.729021,0.652203,0.811400,0.528818,0.693141
0,previous_day,daytime_only,test,8,0.557701,0.609761,0.588883,0.628648,0.568540,0.598425
2,previous_day,no_high_sleep_session,test,8,0.477177,0.548640,0.487092,0.566051,0.459691,0.533333


,feature_timing,experiment_id,subset_name,window,model_family,selected_threshold_from_validation,balanced_accuracy,roc_auc,average_precision,f1,precision,recall
7,same_date,phase1a_003,daytime_only,7,mlp_current_day,0.42,0.843989,0.902264,0.840232,0.821549,0.739394,0.924242
5,same_date,phase1a_002,daytime_only,7,gru,0.34,0.831668,0.904679,0.838141,0.810289,0.703911,0.954545
15,same_date,phase1a_007,daytime_only,14,mlp_current_day,0.43,0.829082,0.877199,0.833664,0.824074,0.754237,0.908163
63,same_date,phase1a_031,no_high_sleep_session,14,mlp_current_day,0.44,0.814567,0.878255,0.838839,0.809302,0.743590,0.887755
1,same_date,phase1a_000,daytime_only,7,bilstm,0.32,0.814436,0.881369,0.792679,0.792079,0.701754,0.909091
3,same_date,phase1a_001,daytime_only,7,cnn_1d,0.50,0.804321,0.897602,0.825660,0.774908,0.755396,0.795455
55,same_date,phase1a_027,no_high_sleep_session,7,mlp_current_day,0.44,0.803280,0.839868,0.731588,0.773234,0.759124,0.787879
71,same_date,phase1a_035,no_high_sleep_session,30,mlp_current_day,0.51,0.800111,0.838686,0.808309,0.829268,0.850000,0.809524
13,same_date,phase1a_006,daytime_only,14,gru,0.28,0.791520,0.882301,0.808663,0.800000,0.676056,0.979592
21,same_date,phase1a_010,daytime_only,30,gru,0.49,0.785899,0.849022,0.882772,0.852941,0.794521,0.920635


saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs\same_date_vs_previous_day_threshold_comparison.csv
saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_outputs\same_date_vs_previous_day_top_test_candidates.csv


In [25]:
# 결과 보고서 초안 생성 - tabulate 없이 Markdown table 생성

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_2A_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning_previous_day"
    / "experiments"
    / "phase_2a_outputs"
)

COMPARISON_PATH = PHASE_2A_OUTPUT_ROOT / "same_date_vs_previous_day_threshold_comparison.csv"
TOP_CANDIDATES_PATH = PHASE_2A_OUTPUT_ROOT / "same_date_vs_previous_day_top_test_candidates.csv"

comparison = pd.read_csv(COMPARISON_PATH, encoding="utf-8-sig")
top_candidates = pd.read_csv(TOP_CANDIDATES_PATH, encoding="utf-8-sig")

test_comparison = comparison[comparison["split"] == "test"].copy()
test_comparison = test_comparison.sort_values(
    ["max_balanced_accuracy", "mean_balanced_accuracy"],
    ascending=False,
)

top_test = top_candidates.sort_values(
    ["balanced_accuracy", "roc_auc", "f1"],
    ascending=False,
).head(10)

best = top_test.iloc[0]

report_path = REPORT_DIR / "deep_learning_subset_ablation_and_previous_day_report.md"


def df_to_markdown_table(df):
    safe_df = df.copy()

    for col in safe_df.columns:
        if pd.api.types.is_float_dtype(safe_df[col]):
            safe_df[col] = safe_df[col].map(lambda x: "" if pd.isna(x) else f"{x:.4f}")
        else:
            safe_df[col] = safe_df[col].map(lambda x: "" if pd.isna(x) else str(x))

    headers = list(safe_df.columns)
    lines = []

    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")

    for _, row in safe_df.iterrows():
        values = [str(row[col]).replace("\n", " ") for col in headers]
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)


top_test_cols = [
    "feature_timing",
    "experiment_id",
    "subset_name",
    "window",
    "model_family",
    "selected_threshold_from_validation",
    "balanced_accuracy",
    "roc_auc",
    "average_precision",
    "f1",
    "precision",
    "recall",
]

lines = []

lines.append("# Deep Learning Subset Ablation And Previous-Day Timing Report")
lines.append("")
lines.append("## Summary")
lines.append("")
lines.append("This report summarizes the first two deep learning data-improvement rounds.")
lines.append("")
lines.append("1. Phase 1A tested same-date feature subsets.")
lines.append("2. Phase 2A tested stricter previous-day features using `target_date - 1 day`.")
lines.append("3. The strongest test performance came from the same-date `daytime_only` feature subset.")
lines.append("4. Previous-day features substantially reduced performance, suggesting that the same-date signal is important and should not be interpreted as strict prior-day forecasting.")
lines.append("")
lines.append("## Best Current Candidate")
lines.append("")
lines.append("| item | value |")
lines.append("| --- | --- |")
lines.append(f"| experiment_id | `{best['experiment_id']}` |")
lines.append(f"| feature_timing | `{best['feature_timing']}` |")
lines.append(f"| subset_name | `{best['subset_name']}` |")
lines.append(f"| window | `{int(best['window'])}` |")
lines.append(f"| model_family | `{best['model_family']}` |")
lines.append(f"| selected threshold | `{best['selected_threshold_from_validation']:.2f}` |")
lines.append(f"| test balanced accuracy | `{best['balanced_accuracy']:.4f}` |")
lines.append(f"| test ROC AUC | `{best['roc_auc']:.4f}` |")
lines.append(f"| test average precision | `{best['average_precision']:.4f}` |")
lines.append(f"| test F1 | `{best['f1']:.4f}` |")
lines.append(f"| test precision | `{best['precision']:.4f}` |")
lines.append(f"| test recall | `{best['recall']:.4f}` |")
lines.append("")
lines.append("## Test Comparison By Feature Timing")
lines.append("")
lines.append(df_to_markdown_table(test_comparison))
lines.append("")
lines.append("## Top Test Candidates")
lines.append("")
lines.append(df_to_markdown_table(top_test[top_test_cols]))
lines.append("")
lines.append("## Interpretation")
lines.append("")
lines.append("The best-performing model is:")
lines.append("")
lines.append("`same_date / daytime_only / window 7 / mlp_current_day`")
lines.append("")
lines.append("This model achieved strong held-out test performance, but it uses features aligned to the sleep target `calendar_date`.")
lines.append("")
lines.append("Earlier date-alignment checks showed that the sleep target `calendar_date` matches the sleep `endTime` date. Therefore, same-date features may include information from the day of sleep completion or information close to the target event.")
lines.append("")
lines.append("The model should be interpreted as a same-date sleep classification model, not a strict previous-day forecasting model.")
lines.append("")
lines.append("## Previous-Day Experiment Result")
lines.append("")
lines.append("The previous-day feature experiment was designed to reduce temporal ambiguity by using only features from `target_date - 1 day`.")
lines.append("")
lines.append("Performance dropped substantially:")
lines.append("")
lines.append("- `same_date + daytime_only`: highest test balanced accuracy around `0.844`")
lines.append("- `previous_day + daytime_only`: highest test balanced accuracy around `0.610`")
lines.append("")
lines.append("This suggests that strictly prior-day features alone are weaker predictors in the current dataset.")
lines.append("")
lines.append("## Practical Conclusion")
lines.append("")
lines.append("For the current project, the most defensible result is to report two tracks:")
lines.append("")
lines.append("1. Best same-date classifier:")
lines.append("   - `same_date / daytime_only / window 7 / mlp_current_day`")
lines.append("   - strong performance")
lines.append("   - not a strict prospective model")
lines.append("")
lines.append("2. Strict previous-day experiment:")
lines.append("   - lower performance")
lines.append("   - useful as a conservative timing-control analysis")
lines.append("")
lines.append("## Limitations")
lines.append("")
lines.append("- Window 30 experiments have small validation/test participant counts and should be interpreted cautiously.")
lines.append("- Some same-date features may occur after the sleep event depending on source timing.")
lines.append("- Participant-level split shift remains a major factor.")
lines.append("- The previous-day dataset loses rows and participants because a prior-day feature row is required.")
lines.append("")
lines.append("## Recommended Next Steps")
lines.append("")
lines.append("1. Use `same_date / daytime_only / window 7 / mlp_current_day` as the current best same-date classifier.")
lines.append("2. Keep `same_date / daytime_only / window 7 / GRU` as the best sequence-model comparison candidate.")
lines.append("3. Report previous-day results as a stricter timing sensitivity analysis.")
lines.append("4. Run participant-level bootstrap confidence intervals for the selected candidates.")
lines.append("5. Update the main pipeline summary notebook once the final reporting structure is fixed.")
lines.append("")

report = "\n".join(lines)
report_path.write_text(report, encoding="utf-8")

print("saved:", report_path)

saved: c:\workSpace\DeepLearnin_sleep\reports\deep_learning_subset_ablation_and_previous_day_report.md
